# Alpaca Weekly Technical Stock Screener

Run the full screener from a notebook: fetch Alpaca daily bars, filter the universe, then screen weekly technical strength using EMA/SMA alignment, AO, volume change, nearby 3M/6M lows, and UO confirmation.


## 1. Setup

Install the project first from the repository root:

```powershell
pip install -e ".[notebook]"
```

Copy `.env.example` to `.env` and add your Alpaca credentials before running the data cells.

In [ ]:
from pathlib import Path
import inspect
import sys

try:
    import pandas as pd
    from dotenv import load_dotenv
    from alpaca.data.enums import DataFeed
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Missing notebook dependencies. From the repository root, run: "
        'pip install -e ".[notebook]"'
    ) from exc

repo_root = Path.cwd()
if not (repo_root / "alpaca_screener").exists():
    repo_root = repo_root.parent
repo_root = repo_root.resolve()

# Prefer this checkout over any installed or previously imported copy.
repo_root_text = str(repo_root)
if repo_root_text in sys.path:
    sys.path.remove(repo_root_text)
sys.path.insert(0, repo_root_text)
for module_name in list(sys.modules):
    if module_name == "alpaca_screener" or module_name.startswith("alpaca_screener."):
        del sys.modules[module_name]

from alpaca_screener.alpaca_data import (
    DEFAULT_EXCHANGES,
    clients_from_environment,
    fetch_daily_bars,
    get_tradable_symbols,
)
from alpaca_screener.strategy import ScreenConfig, _awesome_oscillator, _to_weekly_bars, screen_bars
from alpaca_screener.universe import (
    UniverseFilterConfig,
    load_fundamentals_file,
    filter_universe,
    volume_stats_from_bars,
)

if "ema_fast_window" not in inspect.signature(ScreenConfig).parameters:
    raise RuntimeError(
        "Loaded an old ScreenConfig. Restart the kernel, then rerun this setup cell."
    )

load_dotenv(repo_root / ".env")
repo_root


## 2. Configure the screen

Use `SYMBOLS` for a fast watchlist run, or set it to an empty list to screen active tradable US equities. While experimenting, keep `LIMIT_UNIVERSE` small.

In [ ]:
if "ema_fast_window" not in inspect.signature(ScreenConfig).parameters:
    raise RuntimeError(
        "The notebook kernel has an old alpaca_screener import loaded. "
        "Rerun the Setup cell above, or restart the kernel and run all cells."
    )

SYMBOLS = []
LIMIT_UNIVERSE = 10000  # Example: 500 when SYMBOLS = []
TOP = 25
FEED = DataFeed.IEX  # Use DataFeed.SIP only when your Alpaca account has SIP access.
EXCHANGES = DEFAULT_EXCHANGES  # Defaults to ("NASDAQ", "NYSE").
FUNDAMENTALS_FILE = None  # Example: repo_root / "data" / "fundamentals.csv"

universe_config = UniverseFilterConfig(
    fundamentals_file=str(FUNDAMENTALS_FILE) if FUNDAMENTALS_FILE else None,
    pe_min=None,
    pe_max=None,
    industries=(),  # Example: ("Semiconductors", "Technology")
    market_caps=(),  # Any of: "small", "mid"/"medium", "large"
    cap_mix={},  # Example: {"small": 20, "mid": 30, "large": 50}
    min_30d_share_volume=None,
    min_30d_dollar_volume=None,
    performance_windows=("3m",),  # Any of: "1m", "3m", "6m"
    top_performance_pct=2.0,  # Example: top 2% of performers
    min_performance={},  # Example: {"3m": 25.0}
    max_symbols=None,
)

config = ScreenConfig(
    ema_fast_window=50,
    ema_slow_window=200,
    sma_fast_window=50,
    sma_slow_window=200,
    ao_fast_window=5,
    ao_slow_window=34,
    ao_min=0.5,
    volume_change_pct=5.0,
    low_near_short_weeks=13,
    low_near_long_weeks=26,
    low_near_min_pct=0.0,
    low_near_max_pct=3.0,
    uo_short_window=7,
    uo_medium_window=14,
    uo_long_window=28,
    uo_cross_level=50.0,
    require_uo_cross_up=True,
    min_price=5.0,
    min_weekly_volume=300000,
)
config, universe_config


## 3. Fetch Alpaca data

In [ ]:
data_client, trading_client = clients_from_environment()

if SYMBOLS:
    base_symbols = [symbol.upper() for symbol in SYMBOLS]
else:
    base_symbols = get_tradable_symbols(
        trading_client,
        LIMIT_UNIVERSE,
        exchanges=tuple(EXCHANGES),
    )

if not base_symbols:
    raise RuntimeError(
        f"No base symbols were returned. Check EXCHANGES={EXCHANGES}, "
        "Alpaca asset access, or set SYMBOLS explicitly for a watchlist run."
    )

clock = trading_client.get_clock()
incomplete_session_date = clock.timestamp.date() if clock.is_open else None

if universe_config.needs_fundamentals and not universe_config.fundamentals_file:
    raise RuntimeError(
        "Set FUNDAMENTALS_FILE to a CSV before using P/E, market-cap, cap-mix, "
        "or industry filters. This notebook no longer calls yfinance per symbol."
    )
fundamentals = (
    load_fundamentals_file(universe_config.fundamentals_file)
    if universe_config.needs_fundamentals
    else None
)
volume_stats = None
if universe_config.needs_volume or universe_config.needs_performance:
    volume_bars = fetch_daily_bars(
        data_client,
        base_symbols,
        calendar_days=210,
        feed=FEED,
        incomplete_session_date=incomplete_session_date,
    )
    volume_stats = volume_stats_from_bars(volume_bars)

symbols = filter_universe(
    base_symbols,
    universe_config,
    fundamentals=fundamentals,
    volume_stats=volume_stats,
)
bars_by_symbol = fetch_daily_bars(
    data_client,
    symbols,
    feed=FEED,
    incomplete_session_date=incomplete_session_date,
)

universe_summary = pd.DataFrame(
    [
        {"stage": "base symbols", "count": len(base_symbols)},
        {"stage": "after universe filters", "count": len(symbols)},
        {"stage": "with full bar data", "count": len(bars_by_symbol)},
    ]
)
if len(symbols) < TOP:
    print(
        f"Universe filters returned {len(symbols)} symbols, fewer than TOP={TOP}. "
        "Relax top_performance_pct/min_performance or increase LIMIT_UNIVERSE if needed."
    )
print("Filtered symbols preview:", symbols[:25])
universe_summary

## 4. Run the screener

Results are sorted by `score` from strongest to weakest. The score favors stronger AO, higher weekly volume change, UO strength above the trigger level, and lows that are closer to the 6-month low.


In [ ]:
if not isinstance(bars_by_symbol, dict):
    raise TypeError(
        "bars_by_symbol must be the dictionary returned by fetch_daily_bars. "
        "Rerun the Fetch Alpaca data cell and do not pass base_symbols to screen_bars."
    )

results = screen_bars(bars_by_symbol, config).head(TOP)
results

## 5. Save results

In [ ]:
output_dir = repo_root / "outputs"
output_dir.mkdir(exist_ok=True)

csv_path = output_dir / "screen-results.csv"
json_path = output_dir / "screen-results.json"

results.to_csv(csv_path, index=False)
results.to_json(json_path, orient="records", indent=2)

csv_path, json_path

## 6. Optional: chart the top match

This chart shows weekly close, EMA/SMA 50/200, AO, UO, and the 3-month/6-month lows for the top match.


In [ ]:
import matplotlib.pyplot as plt

if results.empty:
    print("No matches to chart.")
else:
    symbol = results.iloc[0]["symbol"]
    row = results.iloc[0]
    weekly = _to_weekly_bars(bars_by_symbol[symbol]).tail(260).copy()
    weekly["ema_fast"] = weekly["close"].ewm(
        span=config.ema_fast_window,
        adjust=False,
        min_periods=config.ema_fast_window,
    ).mean()
    weekly["ema_slow"] = weekly["close"].ewm(
        span=config.ema_slow_window,
        adjust=False,
        min_periods=config.ema_slow_window,
    ).mean()
    weekly["sma_fast"] = weekly["close"].rolling(config.sma_fast_window).mean()
    weekly["sma_slow"] = weekly["close"].rolling(config.sma_slow_window).mean()
    weekly["ao"] = _awesome_oscillator(
        weekly,
        config.ao_fast_window,
        config.ao_slow_window,
    )

    axes = weekly[["close", "ema_fast", "ema_slow", "sma_fast", "sma_slow"]].plot(
        figsize=(12, 8),
        title=f"{symbol} weekly technical setup",
    )
    axes.axhline(row["low_3m"], color="tab:green", linestyle="--", label="3M low")
    axes.axhline(row["low_6m"], color="tab:red", linestyle=":", label="6M low")
    axes.set_ylabel("Price")
    axes.legend()
    plt.show()

    weekly[["ao"]].plot(figsize=(12, 3), title=f"{symbol} Awesome Oscillator")
    plt.axhline(config.ao_min, color="tab:orange", linestyle="--", label="AO minimum")
    plt.legend()
    plt.show()

    print(
        f"UO crossed from {row['previous_uo']} to {row['uo']}; "
        f"weekly volume changed {row['volume_change_pct']}%."
    )
